In [ ]:
import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta

np.random.seed(42)
random.seed(42)

N_USERS = 1500
N_RESTAURANTS = 200
N_MENU_ITEMS = 1200
N_ORDERS = 25000

CUISINES = ["North Indian", "South Indian", "Chinese", "Italian", "Fast Food", "Biryani"]
CATEGORIES = ["Main", "Beverage", "Dessert", "Side"]
CITY_TIERS = ["Tier1", "Tier2", "Tier3"]

# USERS
users = pd.DataFrame({
    "user_id": range(N_USERS),
    "veg_pref": np.random.choice([0, 1], N_USERS, p=[0.6, 0.4]),
    "price_sensitivity": np.random.choice(["low", "medium", "high"], N_USERS, p=[0.3,0.5,0.2]),
    "fav_cuisine": np.random.choice(CUISINES, N_USERS),
    "city_tier": np.random.choice(CITY_TIERS, N_USERS, p=[0.5,0.3,0.2])
})

# RESTAURANTS
restaurants = pd.DataFrame({
    "restaurant_id": range(N_RESTAURANTS),
    "cuisine": np.random.choice(CUISINES, N_RESTAURANTS),
    "rating": np.round(np.random.uniform(3.5, 4.8, N_RESTAURANTS), 1),
    "price_band": np.random.choice(["low","medium","high"], N_RESTAURANTS, p=[0.4,0.4,0.2]),
    "city_tier": np.random.choice(CITY_TIERS, N_RESTAURANTS, p=[0.5,0.3,0.2])
})

# MENU ITEMS
menu_items = pd.DataFrame({
    "item_id": range(N_MENU_ITEMS),
    "restaurant_id": np.random.choice(restaurants["restaurant_id"], N_MENU_ITEMS),
    "category": np.random.choice(CATEGORIES, N_MENU_ITEMS, p=[0.5,0.25,0.15,0.1]),
    "is_veg": np.random.choice([0,1], N_MENU_ITEMS, p=[0.7,0.3]),
    "price": np.random.randint(80, 500, N_MENU_ITEMS)
})

menu_items = menu_items.merge(
    restaurants[["restaurant_id","cuisine","price_band"]],
    on="restaurant_id", how="left"
)

# Price percentile within restaurant
menu_items["price_percentile_within_restaurant"] = (
    menu_items.groupby("restaurant_id")["price"].rank(pct=True)
)

menu_items["semantic_text"] = (
    menu_items["cuisine"].astype(str) + " " +
    menu_items["category"].astype(str) + " " +
    menu_items["price_band"].astype(str)
)

# ORDER GENERATION
orders = []
start_date = datetime(2025, 1, 1)

for order_id in range(N_ORDERS):

    user = users.sample(1).iloc[0]

    if np.random.rand() < 0.6:
        rest = restaurants[restaurants["cuisine"] == user["fav_cuisine"]].sample(1).iloc[0]
    else:
        rest = restaurants.sample(1).iloc[0]

    order_time = start_date + timedelta(minutes=random.randint(0, 60*24*60))
    hour = order_time.hour
    weekend = int(order_time.weekday() >= 5)

    # Meal bucket
    if 6 <= hour < 11:
        meal_time = "Breakfast"
    elif 11 <= hour < 16:
        meal_time = "Lunch"
    elif 16 <= hour < 22:
        meal_time = "Dinner"
    else:
        meal_time = "LateNight"

    main_items = menu_items[
        (menu_items["restaurant_id"] == rest["restaurant_id"]) &
        (menu_items["category"] == "Main")
    ]

    if user["veg_pref"] == 1:
        main_items = main_items[main_items["is_veg"] == 1]

    if len(main_items) == 0:
        continue

    main_item = main_items.sample(1).iloc[0]
    cart = [main_item]

    def add_on(category, base_prob):
        items = menu_items[
            (menu_items["restaurant_id"] == rest["restaurant_id"]) &
            (menu_items["category"] == category)
        ]
        if len(items) == 0:
            return
        if np.random.rand() < base_prob:
            cart.append(items.sample(1).iloc[0])

    add_on("Beverage", 0.5)
    add_on("Dessert", 0.3)
    add_on("Side", 0.4)

    total_value = sum(item["price"] for item in cart)
    cart_size = len(cart)

    if total_value < 300:
        cart_value_bucket = "Low"
    elif total_value < 700:
        cart_value_bucket = "Medium"
    else:
        cart_value_bucket = "High"

    for item in cart:
        orders.append({
            "order_id": order_id,
            "user_id": user["user_id"],
            "restaurant_id": rest["restaurant_id"],
            "item_id": item["item_id"],
            "category": item["category"],
            "price": item["price"],
            "order_hour": hour,
            "weekend": weekend,
            "timestamp": order_time,
            "meal_time": meal_time,
            "cart_size": cart_size,
            "cart_value_bucket": cart_value_bucket,
            "user_price_sensitivity": user["price_sensitivity"],
            "restaurant_rating": rest["rating"],
            "user_city_tier": user["city_tier"],
            "restaurant_city_tier": rest["city_tier"],
            "order_total_value": total_value
        })

orders_df = pd.DataFrame(orders)

# User behavior features
user_frequency = orders_df.groupby("user_id")["order_id"].nunique()
user_avg_spend = orders_df.groupby("user_id")["order_total_value"].mean()
restaurant_popularity = orders_df.groupby("restaurant_id")["order_id"].nunique()
item_popularity = orders_df.groupby("item_id").size()

orders_df["user_frequency"] = orders_df["user_id"].map(user_frequency)
orders_df["user_avg_spend"] = orders_df["user_id"].map(user_avg_spend)
orders_df["restaurant_popularity"] = orders_df["restaurant_id"].map(restaurant_popularity)
orders_df["item_popularity"] = orders_df["item_id"].map(item_popularity)

print("Total Rows (order-items):", len(orders_df))
print("Unique Orders:", orders_df["order_id"].nunique())
print("Avg items per order:",
      round(len(orders_df) / orders_df["order_id"].nunique(), 2))

print("City Tier Distribution:")
print(orders_df["user_city_tier"].value_counts(normalize=True))

print("Meal Time Distribution:")
print(orders_df["meal_time"].value_counts(normalize=True))

In [ ]:
import numpy as np

csao_rows = []

embeddings = np.random.normal(size=(len(menu_items), 384))
item_embedding_map = dict(zip(menu_items["item_id"], embeddings))

for order_id, group in orders_df.groupby("order_id"):

    main_items = group[group["category"] == "Main"]
    if len(main_items) == 0:
        continue

    main_item = main_items.iloc[0]
    restaurant_id = main_item["restaurant_id"]
    accepted_items = set(group["item_id"].values)

    candidates = menu_items[
        (menu_items["restaurant_id"] == restaurant_id) &
        (menu_items["category"] != "Main")
    ]

    main_embedding = item_embedding_map[main_item["item_id"]]

    for _, candidate in candidates.iterrows():

        candidate_embedding = item_embedding_map[candidate["item_id"]]
        semantic_similarity = np.dot(main_embedding, candidate_embedding) / (
            np.linalg.norm(main_embedding) * np.linalg.norm(candidate_embedding)
        )

        csao_rows.append({
        "order_id": order_id,
        "user_id": main_item["user_id"],
        "timestamp": main_item["timestamp"],
        "order_hour": main_item["order_hour"],
        "weekend": main_item["weekend"],
        "meal_time": main_item["meal_time"],
        "cart_size": main_item["cart_size"],
        "cart_value_bucket": main_item["cart_value_bucket"],
        "user_price_sensitivity": main_item["user_price_sensitivity"],
        "user_city_tier": main_item["user_city_tier"],
        "restaurant_city_tier": main_item["restaurant_city_tier"],
        "same_city_tier": int(main_item["user_city_tier"] == main_item["restaurant_city_tier"]),
        "user_frequency": main_item["user_frequency"],
        "user_avg_spend": main_item["user_avg_spend"],
        "restaurant_popularity": main_item["restaurant_popularity"],
        "candidate_item_id": candidate["item_id"],
        "candidate_category": candidate["category"],
        "candidate_price": candidate["price"],
        "candidate_price_percentile": candidate["price_percentile_within_restaurant"],
        "candidate_popularity": main_item["item_popularity"],
        "semantic_similarity": semantic_similarity,
        "label": int(candidate["item_id"] in accepted_items)
    })

csao_df = pd.DataFrame(csao_rows)

In [ ]:
def reduce_negatives(csao_df, max_negatives=5):
    final_rows = []

    for order_id, group in csao_df.groupby("order_id"):

        positives = group[group["label"] == 1]
        negatives = group[group["label"] == 0]

        if len(positives) == 0:
            continue  # skip empty orders

        # Sample limited negatives
        negatives_sampled = negatives.sample(
            min(max_negatives, len(negatives)),
            random_state=42
        )

        final_group = pd.concat([positives, negatives_sampled])
        final_rows.append(final_group)

    return pd.concat(final_rows).reset_index(drop=True)

In [ ]:
csao_df.to_csv("recommender.csv", index=False)

In [ ]:
csao_df = reduce_negatives(csao_df)

In [ ]:
positives = csao_df[csao_df["label"] == 1]
negatives = csao_df[csao_df["label"] == 0]

negatives = negatives.sample(
    min(len(negatives), len(positives) * NEGATIVE_SAMPLES_PER_ORDER),
    random_state=42
)

csao_df_balanced = pd.concat([positives, negatives]).sample(frac=1, random_state=42)

print("Balanced size:", len(csao_df_balanced))
print("New positive rate:", csao_df_balanced["label"].mean())

Balanced size: 40761
New positive rate: 0.36861215377444123


In [ ]:
csao_df_balanced["timestamp"] = pd.to_datetime(csao_df_balanced["timestamp"])

csao_df_balanced = csao_df_balanced.sort_values("timestamp")

split_time = csao_df_balanced["timestamp"].quantile(0.8)

train_df = csao_df_balanced[
    csao_df_balanced["timestamp"] <= split_time
]

val_df = csao_df_balanced[
    csao_df_balanced["timestamp"] > split_time
]

print("Train rows:", len(train_df))
print("Val rows:", len(val_df))

Train rows: 32611
Val rows: 8150


In [ ]:
DROP_COLS = ["label", "order_id", "user_id", "candidate_item_id", "timestamp"]

X_train = train_df.drop(columns=DROP_COLS)
y_train = train_df["label"]

X_val = val_df.drop(columns=DROP_COLS)
y_val = val_df["label"]

In [ ]:
categorical_cols = [
    "candidate_category",
    "user_price_sensitivity",
    "user_city_tier",
    "restaurant_city_tier",
    "meal_time",
    "cart_value_bucket"
]

for col in categorical_cols:
    X_train[col] = X_train[col].astype("category")
    X_val[col] = X_val[col].astype("category")

In [ ]:
# Time signals
X_train["late_night"] = (X_train["order_hour"] >= 22).astype(int)
X_val["late_night"] = (X_val["order_hour"] >= 22).astype(int)

X_train["is_weekend"] = X_train["weekend"]
X_val["is_weekend"] = X_val["weekend"]

# Category flags
X_train["is_beverage"] = (X_train["candidate_category"] == "Beverage").astype(int)
X_val["is_beverage"] = (X_val["candidate_category"] == "Beverage").astype(int)

X_train["is_dessert"] = (X_train["candidate_category"] == "Dessert").astype(int)
X_val["is_dessert"] = (X_val["candidate_category"] == "Dessert").astype(int)

# Price signals
X_train["expensive_candidate"] = (X_train["candidate_price"] > 300).astype(int)
X_val["expensive_candidate"] = (X_val["candidate_price"] > 300).astype(int)

X_train["high_price_sensitive"] = (X_train["user_price_sensitivity"] == "high").astype(int)
X_val["high_price_sensitive"] = (X_val["user_price_sensitivity"] == "high").astype(int)

# Powerful interaction
X_train["price_sensitive_x_expensive"] = (
    X_train["high_price_sensitive"] * X_train["expensive_candidate"]
)
X_val["price_sensitive_x_expensive"] = (
    X_val["high_price_sensitive"] * X_val["expensive_candidate"]
)

In [ ]:
import lightgbm as lgb
from lightgbm import LGBMRanker

model = LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=800,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1,
    reg_lambda=2
)
group = train_df.groupby("order_id").size().values
model.fit(
    X_train,
    y_train,
    group=group
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004682 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1088
[LightGBM] [Info] Number of data points in the train set: 32611, number of used features: 24


LGBMRanker(colsample_bytree=0.8, learning_rate=0.03, metric='ndcg',
           min_child_samples=50, n_estimators=800, num_leaves=63,
           objective='lambdarank', reg_alpha=1, reg_lambda=2, subsample=0.8)

In [ ]:
val_df["pred"] = model.predict(X_val)

/tmp/ipykernel_644/971891497.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_df["pred"] = model.predict(X_val)


In [ ]:
def precision_at_k(df, k=5):
    precisions = []

    for order_id, group in df.groupby("order_id"):
        top_k = group.sort_values("pred", ascending=False).head(k)
        precisions.append(top_k["label"].sum() / k)

    return np.mean(precisions)

precision_at_k(val_df, k=5)

np.float64(0.25222502099076405)

In [ ]:
print("Overall positive rate:", csao_df["label"].mean())

Overall positive rate: 0.36861215377444123


In [ ]:
print(csao_df.groupby("order_id")["label"].sum().describe())

count    11848.000000
mean         1.268147
std          0.485729
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          3.000000
Name: label, dtype: float64


In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(val_df["label"], val_df["pred"])
print("Validation AUC:", auc)

Validation AUC: 0.8128275040806744


In [ ]:
def recall_at_k(df, k=5):
    recalls = []
    for order_id, group in df.groupby("order_id"):
        top_k = group.sort_values("pred", ascending=False).head(k)
        total_positives = group["label"].sum()
        if total_positives > 0:
            recalls.append(top_k["label"].sum() / total_positives)
    return np.mean(recalls)

print("Recall@5:", recall_at_k(val_df, 5))

Recall@5: 0.9929331094318501


In [ ]:
from sklearn.metrics import ndcg_score
import numpy as np

def compute_ndcg(df, k=5):
    ndcgs = []

    for order_id, group in df.groupby("order_id"):
        if len(group) < 2:
            continue
        true = group["label"].values.reshape(1, -1)
        scores = group["pred"].values.reshape(1, -1)
        ndcgs.append(ndcg_score(true, scores, k=k))

    if len(ndcgs) == 0:
        return 0.0

    return np.mean(ndcgs)

print("NDCG@5:", compute_ndcg(val_df, 5))

NDCG@5: 0.8222192913895857


In [ ]:
val_df_baseline = val_df.copy()

item_popularity = train_df.groupby("candidate_item_id")["label"].mean()

val_df_baseline["pred"] = (
    val_df_baseline["candidate_item_id"]
    .map(item_popularity)
    .fillna(0)
)

print("Baseline Precision@5:",
      precision_at_k(val_df_baseline, 5))

Baseline Precision@5: 0.2510495382031906


In [ ]:
def acceptance_rate(df, k=5):
    accepted = 0
    total = 0
    for order_id, group in df.groupby("order_id"):
        top_k = group.sort_values("pred", ascending=False).head(k)
        accepted += top_k["label"].sum()
        total += k
    return accepted / total

print("Acceptance Rate@5:", acceptance_rate(val_df, 5))

Acceptance Rate@5: 0.25222502099076405


In [ ]:
def compute_aov_lift(df, k=5):
    lifts = []
    for order_id, group in df.groupby("order_id"):
        top_k = group.sort_values("pred", ascending=False).head(k)
        lifts.append(top_k[top_k["label"] == 1]["candidate_price"].sum())
    return np.mean(lifts)

print("Projected AOV Lift:", compute_aov_lift(val_df, 5))

Projected AOV Lift: 363.93828715365237


In [ ]:
def category_diversity(df, k=5):
    diversities = []
    for order_id, group in df.groupby("order_id"):
        top_k = group.sort_values("pred", ascending=False).head(k)
        diversities.append(top_k["candidate_category"].nunique())
    return np.mean(diversities)

print("Avg Category Diversity@5:", category_diversity(val_df, 5))

Avg Category Diversity@5: 2.0965575146935347


In [ ]:
import time

sample = X_val.sample(1000)

start = time.time()
_ = model.predict(sample)
end = time.time()

print("Inference time for 1000 rows:", (end - start)*1000, "ms")
print("Approx per request latency:", ((end - start)*1000)/1000, "ms")

Inference time for 1000 rows: 170.4566478729248 ms
Approx per request latency: 0.1704566478729248 ms
